# 11 - Utilities Reference

> **Related**: This notebook demonstrates the use of sqlseed's internal utility classes, including MetricsCollector, sql_safe, Progress, Logger, and schema_helpers.

## What You Will Learn

- MetricsCollector performance metrics
- sql_safe SQL injection protection
- Progress multi-backend progress system
- Logger structlog logging
- schema_helpers AUTOINCREMENT detection

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 01 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| **→ 11** | **Utilities Reference** | **Utils** | **01** |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [ ]:
from __future__ import annotations

# Run from examples/notebooks. Install from the repository root in one resolution:
# python -m pip install -e ".[dev,all]" -e "./plugins/sqlseed-cli" \
#   -e "./plugins/sqlseed-ai[dev,mcp]" -e "./plugins/mcp-server-sqlseed" -e "./plugins/sqlseed-web[dev]"
import os
import sqlite3
import sys
import tempfile
from contextlib import closing
from pathlib import Path

import sqlseed
from sqlseed import connect

sys.path.insert(0, str(Path("..").resolve()))  # build_demo_db only
from build_demo_db import build

# Keep this object alive across cells. No existing database is opened or rebuilt.
_demo_directory = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-")
demo_root = Path(_demo_directory.name)
os.environ["SQLSEED_CACHE_DIR"] = str(demo_root / "cache")
db_path = build(demo_root / "demo.db")


def require(condition, message):
    """Stop the tutorial if an expected outcome did not occur."""
    if not condition:
        raise RuntimeError(message)


def check_generation(result, count):
    """Check errors and generated row count before showing success."""
    require(not result.errors and result.count == count, f"Generation failed: {result.errors}; count={result.count}")


def read_rows(database, sql):
    """Read actual persisted values using a fixed tutorial query."""
    # The queries below are fixed tutorial SQL, never external identifiers.
    with closing(sqlite3.connect(database)) as connection:
        return connection.execute(sql).fetchall()


with connect(str(db_path), provider="faker") as orch:
    for table, count in (("organizations", 5), ("members", 20), ("projects", 10), ("tags", 8)):
        check_generation(orch.fill_table(table, count=count, seed=42, skip_ai=True), count)

print(f"sqlseed {sqlseed.__version__} | Temporary database: {db_path}")

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Internal Utilities | `src/sqlseed/_utils/` | `progress.py`, `sql_safe.py`, `metrics.py` |

## 1. MetricsCollector Performance Metrics

MetricsCollector collects and aggregates performance metrics, supporting filtering and statistics by name.

In [ ]:
from sqlseed._utils.metrics import MetricsCollector

metrics = MetricsCollector()

metrics.record("fill_time", 1.234)
metrics.record("fill_time", 0.567)
metrics.record("fill_time", 2.891)
metrics.record("insert_count", 1000)
metrics.record("insert_count", 2000)

print("MetricsCollector API:")
print("  record('fill_time', 1.234) → records a metric")
print(f"  get_entries('fill_time') → {len(metrics.get_entries('fill_time'))} entries")
print(f"  get_entries() → {len(metrics.get_entries())} entries (all)")

summary = metrics.summary()
print("\n  summary():")
for name, stats in summary.items():
    print(f"    {name}: count={stats['count']}, avg={stats['avg']:.3f}, min={stats['min']:.3f}, max={stats['max']:.3f}")

require(summary["fill_time"]["count"] == 3, "Metrics count differs")
require(summary["insert_count"]["max"] == 2000, "Metrics maximum differs")
metrics.clear()
print(f"\n  clear() → {len(metrics.get_entries())} entries (cleared)")

## 2. sql_safe SQL Injection Protection

The sql_safe module provides SQL identifier escaping and validation to prevent SQL injection.

In [ ]:
from sqlseed._utils.sql_safe import build_insert_sql, quote_identifier, validate_table_name

print("sql_safe — SQL injection protection:\n")

print(f"  quote_identifier('table_name') → {quote_identifier('table_name')}")
print("  quote_identifier('table\"name') → " + quote_identifier('table"name'))

print(f"\n  validate_table_name('users') → {validate_table_name('users')}")

sql = build_insert_sql("users", ["name", "email", "age"])
print("\n  build_insert_sql('users', ['name', 'email', 'age']):")
print(f"    {sql}")

## 3. Progress Multi-Backend System

sqlseed uses the Strategy Pattern to implement cross-environment progress bars:
- **Terminal**: `RichProgressBackend` (based on Rich)
- **Jupyter**: `TqdmNotebookBackend` (based on tqdm.auto)
- **No UI**: `NullProgressBackend` (zero overhead)

`create_progress()` auto-detects the runtime environment and selects the appropriate backend.

In [ ]:
from sqlseed._utils.progress import create_progress

# create_progress() returns a multi-backend Progress context manager
# Used internally by fill_table for batch progress display
progress = create_progress()
print(f"Progress type: {type(progress).__name__}")
print()
print("Usage in fill_table:")
print("  with create_progress() as progress:")
print('      task = progress.add_task("Filling...", total=count)')
print("      for batch in data_stream.generate():")
print("          progress.update(task, advance=len(batch))")
print()
print("Columns: Spinner, Bar, Percentage, Count, Speed, Time Remaining")

## 4. Logger structlog Logging

sqlseed uses structlog for structured logging; the log level is controlled via `GeneratorConfig.log_level`.

In [ ]:
from sqlseed.config.models import GeneratorConfig

print("Logger — structlog logging:")
print("  sqlseed uses structlog for structured logging")
print("  Log level is controlled via GeneratorConfig.log_level")
print("  Default: INFO")

config = GeneratorConfig(db_path=str(db_path), log_level="DEBUG")
print(f"\n  GeneratorConfig(log_level='DEBUG') → {config.log_level}")

## 5. schema_helpers AUTOINCREMENT Detection

schema_helpers provides AUTOINCREMENT primary key detection, used by ColumnMapper Level 1 to decide whether to skip generation.

In [ ]:
print("schema_helpers — AUTOINCREMENT detection:")
print("  detect_autoincrement(execute_fn, table_name, column_name) → checks an auto-increment primary key")
print("  Used by ColumnMapper Level 1 to decide whether to skip generation")

with sqlseed.connect(str(db_path)) as orch:
    col_info = orch.get_column_info("organizations")
    for col in col_info:
        if col.is_primary_key:
            print(f"\n  {col.name}: is_pk={col.is_primary_key}, is_autoincrement={col.is_autoincrement}")

## 6. ExpressionEngine Expression Engine

ExpressionEngine uses simpleeval to execute safe Python expressions, using a whitelist exposed as `ExpressionEngine.SAFE_FUNCTIONS`.

In [ ]:
from sqlseed.core.expression import ExpressionEngine

engine = ExpressionEngine()

# Basic arithmetic
print(f"2 + 3 = {engine.evaluate('a + b', {'a': 2, 'b': 3})}")

# String operations
print(f"upper: {engine.evaluate('name.upper()', {'name': 'hello'})}")

print(f"Safe function registry: {len(engine.SAFE_FUNCTIONS)} names")
# Representative safe functions
print(f"abs(-5): {engine.evaluate('abs(x)', {'x': -5})}")
print(f"len: {engine.evaluate('len(s)', {'s': 'abc'})}")
print(f"min(1,2,3): {engine.evaluate('min(a,b,c)', {'a': 1, 'b': 2, 'c': 3})}")

## ✅ Summary

| Tool | Function | Status |
|---|---|---|
| MetricsCollector | Performance metrics | ✅ |
| sql_safe | SQL injection protection | ✅ |
| Progress | Multi-backend progress system | ✅ |
| Logger | structlog logging | ✅ |
| schema_helpers | AUTOINCREMENT detection | ✅ |

**Next**: [12-testing-patterns.ipynb](12-testing-patterns.ipynb) — Testing Integration Patterns

In [ ]:
require(len(read_rows(db_path, "SELECT member_id FROM members")) == 20, "The examples changed unrelated members")
print("Tutorial operations completed with the original 20 demo members preserved.")